# CR-1 G1 土台の道 ＋ G1b 費用の見込み（Claude API・CPU）

凍結した CR-1 の計画の G1 と G1b です。

- **G1**：100 のテンプレートそれぞれで、候補（段2 で固定した最大 3 問）を順に、Haiku 4.5（温度 0）で 1 本ずつ解かせ、条件（答えが読める・答えより前の単位が 30〜80 個・最後まで書き終えた）を満たした最初の 1 問を使います。使えた問題の数 N が 70 以上なら先へ。
- **G1b**：G1 の実績から本番の費用を計画の式で見込み、上限 $90 を超えるなら、テンプレート番号の大きい方から外します。外したあと N ≥ 70 なら GO。
- 問題文・応答はどこにも表示しません（数・ID・ハッシュだけ）。文字列は Drive の非公開のファイルにだけ保存します。

**使い方**
1. ランタイムは **CPU** で大丈夫です。
2. シークレットは `ANTHROPIC_API_KEY` と `GITHUB_TOKEN`。
3. セル1 → セル2 → セル3 の順に実行（セル2 は 10〜20 分くらい）。途中で切れても、もう一度セル1 → セル2 を実行すれば、終わったテンプレートは飛ばして続きから回ります。
4. 費用は $0.5 前後の見込み（$3 を超えそうなら止まります）。

結果は Drive の `DECISION_POINTS/cr1/G1/CR1_G1.json` に入ります。終わったら Claude に教えてね。

- **通知**：スマホの ntfy アプリで `godpork-b06080cfb3cdfa8f` を購読しておくと、始まったとき・終わったとき・エラーで止まったときに届きます。ランタイムごと切れたときは届かないので、そのときは Colab を見てください。
- **終わったら自動でランタイムを切ります**（CU の節約）。Drive への保存を確定してから切るので、結果は残ります。エラーで止まったときは切らずに残します。切りたくないときは、最初に `AUTO_DISCONNECT = False` を実行してください。

In [ ]:
# Cell 1 — Drive, exact head, pinned SDK, key (never printed), notifications.
from google.colab import drive, userdata
from pathlib import Path
import os, shutil, stat, subprocess, sys, json, time, datetime, hashlib, threading

drive.mount("/content/drive")
EXPECTED_HEAD = "10977e164564b2ee8eb87b614b3bb23222fbc250"
BRANCH = "decision-points"
REPO_URL = "https://github.com/chichichi2001/Pork.git"
REPO = Path("/content/Pork")
OUT = Path("/content/drive/MyDrive/DECISION_POINTS/cr1/G1")
OUT.mkdir(parents=True, exist_ok=True)

def get_secret(*names):
    for name in names:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
        if value and str(value).strip():
            return str(value).strip()
    return None

token = get_secret("GITHUB_TOKEN", "GH_TOKEN")
assert token, "Colab Secrets に GITHUB_TOKEN を登録してください。"
askpass = Path("/tmp/pork_git_askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in\n  *Username*) echo "x-access-token" ;;\n  *Password*) echo "$GITHUB_TOKEN" ;;\n  *) echo "" ;;\nesac\n')
askpass.chmod(askpass.stat().st_mode | stat.S_IXUSR)
env = dict(os.environ, GITHUB_TOKEN=token, GIT_ASKPASS=str(askpass), GIT_TERMINAL_PROMPT="0")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True, env=env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--detach", EXPECTED_HEAD], check=True)
head = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
assert head == EXPECTED_HEAD, (head, EXPECTED_HEAD)
token = None
env.pop("GITHUB_TOKEN", None)
for f in ("notes/cr1_lock_pre_g0.md", "notes/cr1_prompt_record.md", "notes/cr1_g0_result.md"):
    assert (REPO / f).exists(), f + " がありません"
sys.path.insert(0, str(REPO))
from pork import cr1_requests as RQ
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "anthropic==" + RQ.SDK_VERSION], capture_output=True, text=True)
assert r.returncode == 0, r.stderr[-2000:]
import anthropic
assert anthropic.__version__ == RQ.SDK_VERSION, anthropic.__version__
_key = get_secret("ANTHROPIC_API_KEY")
assert _key, "Colab のシークレットに ANTHROPIC_API_KEY を登録してね。"
client = anthropic.Anthropic(api_key=_key, max_retries=0)
del _key
print("exact head:", head, "| anthropic", anthropic.__version__)

# 通知（ntfy）：スマホの ntfy アプリでこの名前を購読しておくと、始まったとき・終わったとき・止まったときに届きます。中身は一言だけ（データは送りません）
import urllib.request
NTFY_TOPIC = 'godpork-b06080cfb3cdfa8f'
NB_LABEL = 'CR-1 G1 土台の道＋G1b（API・CPU）'   # どのノートブックからの通知かの名札
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=('[' + NB_LABEL + '] ' + msg).encode('utf-8'),
                                     headers={'Title': 'Pork - CR-1 G1', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        notify('止まりました：エラーが出ています。Colab を見てください', tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

In [ ]:
# Cell 2 — G1: baseline paths (temperature 0), resumable; texts only in the PRIVATE file.
from concurrent.futures import ThreadPoolExecutor, as_completed
from pork import dp7_matched as d7
from pork import cr1_items as CI
from pork import cr1_analysis as A
from pork.quality_pork_q2a_gsm8k import parse_gold
import decimal

CAND_SHA = "b6f6375aed95831de0098a5138bb87f4101fe9c6670598f23e5e8055eaac8511"   # 段2
G1_CAP = 3.0
rows, DATA_ID = d7.d2.load_rows_from_hub()
chain = CI.chain_and_candidates(rows)
assert CI.candidates_sha256(chain) == CAND_SHA, "候補の一覧が段2 と違います。止めます。"
del rows

STATE = OUT / "CR1_G1_state.json"
PRIV = OUT / "CR1_G1_baselines_PRIVATE.json"
S = json.loads(STATE.read_text()) if STATE.exists() else {"templates": {}, "spent_usd": 0.0}
P = json.loads(PRIV.read_text()) if PRIV.exists() else {}
lock = threading.Lock()

def save():
    STATE.write_text(json.dumps(S, ensure_ascii=False, indent=1))
    PRIV.write_text(json.dumps(P, ensure_ascii=False))

def baseline(question):
    params = RQ.baseline_params(question)
    for attempt in range(6):
        try:
            return client.messages.create(**params)
        except (anthropic.RateLimitError, anthropic.APIConnectionError, anthropic.InternalServerError, anthropic.APITimeoutError):
            if attempt == 5:
                raise
            time.sleep(2 ** attempt * 3)

def run_template(c):
    tid = c["template"]
    tried = []
    for j, it in enumerate(c["candidates"]):
        with lock:
            if S["spent_usd"] > G1_CAP - 0.2:
                raise RuntimeError("G1 の費用の見張り（$3）に近づいたので止めました")
        m = baseline(it["question"])
        text = "".join(b.text for b in m.content if getattr(b, "type", "") == "text")
        chk = A.baseline_check(text, m.stop_reason)
        ans = chk["answer"]
        try:
            correct = ans is not None and decimal.Decimal(ans) == parse_gold(it["answer"])
        except decimal.InvalidOperation:
            correct = False
        rec = {"cand": j, "id": int(it["id"]), "instance": str(it["instance"]), "ok": chk["ok"], "reason": chk["reason"],
               "L": chk["L"], "stop_reason": m.stop_reason, "prompt_tokens": m.usage.input_tokens,
               "output_tokens": m.usage.output_tokens, "correct": bool(correct),
               "text_sha256": hashlib.sha256(text.encode("utf-8")).hexdigest()}
        tried.append(rec)
        with lock:
            S["spent_usd"] += (m.usage.input_tokens * RQ.PRICE_IN + m.usage.output_tokens * RQ.PRICE_OUT) / 1e6
        if chk["ok"]:
            with lock:
                P[str(tid)] = {"id": int(it["id"]), "instance": str(it["instance"]), "text": text}
            break
    with lock:
        S["templates"][str(tid)] = {"tried": tried, "used": next((r["cand"] for r in tried if r["ok"]), None)}
        save()
    return tid

todo = [c for c in chain if str(c["template"]) not in S["templates"]]
print("残りのテンプレート:", len(todo), "/", len(chain))
done = 0
with ThreadPoolExecutor(max_workers=8) as ex:
    futs = [ex.submit(run_template, c) for c in todo]
    for f in as_completed(futs):
        f.result()
        done += 1
        if done % 10 == 0:
            print(f"{done}/{len(todo)} テンプレート終わり（費用 ${S['spent_usd']:.3f}）")
save()
print("G1 の呼び出しが終わりました。費用（見込み）: $%.3f" % S["spent_usd"])

In [ ]:
# Cell 3 — G1 gate (N >= 70) and G1b (cost estimate, drop templates by largest id); numbers only.
T = S["templates"]
assert len(T) == 100, "まだ終わっていないテンプレートがあります。セル2 をもう一度実行してね。"
used = []
reasons = {}
for tid in sorted(T, key=int):
    t = T[tid]
    for r in t["tried"]:
        if not r["ok"]:
            reasons[r["reason"]] = reasons.get(r["reason"], 0) + 1
    if t["used"] is not None:
        r = t["tried"][t["used"]]
        used.append({"template": int(tid), "id": r["id"], "instance": r["instance"], "cand": r["cand"], "L": r["L"],
                     "prompt_tokens": r["prompt_tokens"], "output_tokens": r["output_tokens"], "correct": r["correct"],
                     "text_sha256": r["text_sha256"]})
N1 = len(used)
summary = {"schema": "cr1-g1-g1b-v1", "code_head": head, "sdk": anthropic.__version__, "prompt_sha256": RQ.prompt_sha256(),
           "dataset": DATA_ID, "candidates_sha256": CAND_SHA, "g1_spent_usd": S["spent_usd"],
           "N_after_G1": N1, "failed_candidates_by_reason": reasons,
           "templates_without_item": sorted(int(t) for t in T if T[t]["used"] is None),
           "candidate_index_used": {str(k): sum(1 for u in used if u["cand"] == k) for k in range(3)},
           "baseline_accuracy": (sum(u["correct"] for u in used) / N1) if N1 else None,
           "L_mean": (sum(u["L"] for u in used) / N1) if N1 else None,
           "L_min": min((u["L"] for u in used), default=None), "L_max": max((u["L"] for u in used), default=None)}
if N1 < 70:
    summary["verdict"] = "STOPPED_BEFORE_MAIN_RUN"
    summary["why"] = "G1: N < 70"
else:
    g1b = RQ.g1b_select(used)
    final = [u for u in used if u["template"] in set(g1b["final_templates"])]
    summary.update(G1b=g1b, N_final=len(final), final_items_ids_sha256=RQ.items_ids_sha256(final),
                   final_items=[{k: u[k] for k in ("template", "id", "instance", "L", "text_sha256")} for u in final],
                   verdict=g1b["verdict"])
(OUT / "CR1_G1.json").write_text(json.dumps(summary, ensure_ascii=False, indent=1))
print(summary["verdict"], "| N（G1 のあと）:", N1, "| N（最終）:", summary.get("N_final"),
      "| 本番の費用の見込み: $%.2f" % summary["G1b"]["estimate"]["usd"] if "G1b" in summary else "")
print("外れた理由の数:", reasons, "| 土台の道の正解率: %.3f" % summary["baseline_accuracy"] if N1 else "")
notify('終わりました：G1 と G1b の結果を Drive に書きました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    print('Drive への書き込みを確定してから、ランタイムを切ります（CU の節約。止めたいときは AUTO_DISCONNECT = False）')
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount()
    time.sleep(3)
    _runtime.unassign()